In [ ]:
# coding=utf-8
# Copyright 2026 The Google Research Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Setup

In [ ]:
# @title Install dependencies
!pip install -q google-genai jinja2 tqdm pandas kagglehub

In [ ]:
# @title Clone the repository and set up imports
import os
import sys

# Clone the google-research repo if not already present.
if not os.path.exists('google-research'):
  !git clone --depth 1 --filter=blob:none --sparse https://github.com/google-research/google-research.git
  !cd google-research && git sparse-checkout set behavioral_dispositions

# Add to path so we can import the modules.
if 'google-research' not in sys.path:
  sys.path.insert(0, 'google-research')

In [ ]:
# @title Imports
import pandas as pd
from google import genai

from behavioral_dispositions.src import action_generation
from behavioral_dispositions.src import judge_grading
from behavioral_dispositions.src import alignment_analysis
from behavioral_dispositions.src import gemini_runner

## Configuration

In [ ]:
# @title Set your API key { run: "auto" }
# You can get an API key from https://aistudio.google.com/apikey
# Option 1: Set via Colab secrets (recommended)
try:
  from google.colab import userdata
  API_KEY = userdata.get('GOOGLE_API_KEY')
  print('API key loaded from Colab secrets.')
except (ImportError, userdata.SecretNotFoundError):
  # Option 2: Paste directly (less secure)
  API_KEY = ''  # @param {type:"string"}
  if not API_KEY:
    raise ValueError(
        'Please set your API key. You can get one from '
        'https://aistudio.google.com/apikey'
    )

client = genai.Client(api_key=API_KEY)

In [ ]:
# @title Evaluation Configuration { run: "auto" }

# The model to evaluate.
EVAL_MODEL_NAME = 'gemini-3-flash-preview'  # @param {type:"string"}

# The judge model (used for grading responses).
JUDGE_MODEL_NAME = 'gemini-3-flash-preview'  # @param {type:"string"}

# Number of replications per scenario (used to estimate confidence).
NUM_REPLICATIONS = 6  # @param {type:"integer"}

# Maximum concurrent API calls.
MAX_WORKERS = 30  # @param {type:"integer"}

# Sampling temperature for the evaluated model.
TEMPERATURE = 1.0  # @param {type:"number"}

# Maximum number of rows to sample from the dataset, or -1 to include all rows.
MAX_ROWS = 500  # @param {type:"integer"}

## Load Dataset

In [ ]:
# @title Download and load the SJT dataset from Kaggle
import kagglehub

# Download the dataset.
dataset_path = kagglehub.dataset_download(
    'conferencerelease/alignment-of-behavioral-dispositions-in-llms'
)
print(f'Dataset downloaded to: {dataset_path}')

# Find the CSV file.
csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
assert len(csv_files) == 1, f'Expected 1 CSV file, found {csv_files}'
csv_path = os.path.join(dataset_path, csv_files[0])

# Load and display.
dataset_df = pd.read_csv(csv_path)
print(f'Loaded {len(dataset_df)} scenarios.')
print(f'Traits: {dataset_df["trait"].unique().tolist()}')
print(f'Columns: {dataset_df.columns.tolist()}')

# Filter rows with no consensus, as this notebook only computes the results
# needed for Figure 4 of the paper.
dataset_df = dataset_df[
    (dataset_df['human_aggregate_score'] <= 2)
    | (dataset_df['human_aggregate_score'] >= 8)
]
print(f'Filtered to {len(dataset_df)} scenarios with consensus.')

if MAX_ROWS > 0:
  dataset_df = dataset_df.sample(n=MAX_ROWS, random_state=42)
  print(f'Sampled {len(dataset_df)} scenarios.')

dataset_df.head()

In [ ]:
# @title Prepare dataset index
# Add a numeric index if not present, or use existing 'id' column.
if 'id' in dataset_df.columns:
  dataset_df = dataset_df.set_index('id')
else:
  dataset_df.index.name = 'id'

print(f'Dataset ready with {len(dataset_df)} scenarios.')
dataset_df.head()

## Stage 1: Action Generation

Run the evaluated model on each SJT scenario multiple times to estimate
its behavioral tendency. Each replication produces a free-text response.

In [ ]:
# @title Run action generation
eval_runner = gemini_runner.GeminiRunner(
    client=client, model_name=EVAL_MODEL_NAME
)

action_gen_df = action_generation.run_action_generation(
    runner=eval_runner,
    dataset_df=dataset_df,
    num_replications=NUM_REPLICATIONS,
    max_workers=MAX_WORKERS,
    temperature=TEMPERATURE,
)

print(f'Generated responses for {len(action_gen_df)} scenarios.')
print(f'Total responses: {action_gen_df["responses"].apply(len).sum()}')
action_gen_df.head()

In [ ]:
# @title Inspect sample responses
sample = action_gen_df.iloc[0]
print(f'Scenario ID: {sample["input_id"]}')
print(f'User Message: {sample["scenario_text"][:200]}...')
print(f'\nModel Responses ({len(sample["responses"])} replications):')
for i, resp in enumerate(sample['responses']):
  print(f'  [{i+1}] {resp[:150]}')

## Stage 2: Judge Grading

A judge LLM reads each model response and classifies it as recommending
the AGREE action, the OPPOSE action, or neither.

In [ ]:
# @title Run judge grading
judge = gemini_runner.GeminiRunner(
    client=client, model_name=JUDGE_MODEL_NAME
)

grading_df = judge_grading.run_judge_grading(
    action_generation_df=action_gen_df,
    dataset_df=dataset_df,
    judge_runner=judge,
    max_workers=MAX_WORKERS,
)

# Summary statistics.
total_graded = grading_df['grades'].apply(len).sum()
total_responses = action_gen_df['responses'].apply(len).sum()
print(f'Successfully graded {total_graded} / {total_responses} responses.')
grading_df.head()

In [ ]:
# @title Inspect grade distribution
import collections

all_grades = [g for grades in grading_df['grades'] for g in grades]
grade_counts = collections.Counter(all_grades)
print('Grade distribution:')
print(f'  OPPOSE (1): {grade_counts.get(1, 0)}')
print(f'  AGREE  (2): {grade_counts.get(2, 0)}')
print(f'  Total:      {len(all_grades)}')

## Stage 3: Alignment Analysis

Compare the model's behavior to human consensus. The **Directional Alignment**
metric measures whether the model's majority action choice matches the
direction of human agreement.

Results are partitioned by human consensus strength:
- **10 Consensus**: Perfect unanimity (all 10 annotators agree).
- **[9,10) Consensus**: High consensus (9 out of 10 agree).
- **[8,9) Consensus**: Substantial consensus (8 out of 10 agree).

In [ ]:
# @title Compute alignment metrics
detailed_df, summary_df = alignment_analysis.run_alignment_analysis(
    grading_df=grading_df,
    dataset_df=dataset_df,
)

print('=== Directional Alignment (%) by Trait and Consensus Level ===')
print()
summary_df

In [ ]:
# @title Overall alignment
if not detailed_df.empty and 'alignment' in detailed_df.columns:
  overall = detailed_df['alignment'].mean() * 100
  print(f'Overall Directional Alignment: {overall:.1f}%')
  print()

  # Per-trait breakdown.
  per_trait = (
      detailed_df.groupby('trait')['alignment']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'Alignment (%)', 'count': 'N'})
  )
  per_trait['Alignment (%)'] = (per_trait['Alignment (%)'] * 100).round(1)
  print('Per-trait alignment:')
  print(per_trait)
else:
  print('No alignment results to display.')

In [ ]:
# @title Save results
output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

action_gen_df.to_csv(
    os.path.join(output_dir, 'action_generation.csv'), index=False
)
# Save grades (drop the GradedResponse objects for CSV compatibility).
grading_save = grading_df.drop(columns=['graded_responses'])
grading_save.to_csv(
    os.path.join(output_dir, 'grading.csv'), index=False
)
summary_df.to_csv(
    os.path.join(output_dir, 'alignment_summary.csv'), index=False
)

print(f'Results saved to {output_dir}/')